# Road Geometry Linear Regression — First Steps

This notebook runs a simple linear regression on the full `data/road_segments.csv` file. It loads and checks the data, plots the two measurements, fits a line, and evaluates it on held-out rows.

- **X (predictor):** `SEGMENT_LENGTH_KM` — recorded road-segment length in kilometres.
- **Y (target):** `Shape__Length` — GIS geometry length in source units.

> The units of `Shape__Length` are not confirmed by the source, so it is treated as "source units" and not assumed to be metres (see `results/findings.md`). The first fit below uses all eligible records and is **not** a test-set evaluation; sections 6 onwards hold out a test set.

## 1. Import the tools

This notebook uses pandas to read the CSV, NumPy to check numeric values, and Apache ECharts (through the small helper module `src/charts.py`) for interactive charts. ECharts is loaded from a CDN, so charts need an internet connection. The regression mathematics (least-squares fit, predictions, and error metrics) is implemented in C++ and imported as the `regression_cpp` Python module. Build it once from the repository root with `python setup.py build_ext --inplace`.

In [26]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Python looks for modules in sys.path. Add the repository root (holds the compiled
# regression_cpp extension) and src/ (holds charts.py) so the imports below work
# whether the notebook is opened from the repo root or from notebooks/.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "src" / "charts.py").is_file():
        sys.path.insert(0, str(root))
        sys.path.insert(0, str(root / "src"))
        break

try:
    import regression_cpp
except ImportError as error:
    raise ImportError(
        "Build the C++ module first: python setup.py build_ext --inplace"
    ) from error

import charts
import importlib

importlib.reload(charts)  # pick up edits to src/charts.py without restarting the kernel

# Labels and a caveat reused by every chart so wording stays consistent.
X_LABEL = "Segment length (km)"
Y_LABEL = "GIS shape length (source units)"
UNIT_NOTE = "Shape__Length units are unconfirmed."

## 2. Load the project CSV

The path check supports running this notebook either from the repository root or from its `notebooks` folder. The CSV is read with pandas' CSV parser so quoted fields such as `geometry_json` are handled correctly.

In [27]:
# Try data/ under the current folder, then under its parent (notebook opened from notebooks/).
csv_name = "road_segments.csv"
csv_candidates = [
    Path.cwd() / "data" / csv_name,
    Path.cwd().parent / "data" / csv_name,
]
csv_path = next((path for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    searched = "\n".join(str(path) for path in csv_candidates)
    raise FileNotFoundError(
        f"Could not find {csv_name}. Searched these locations:\n{searched}"
    )

# low_memory=False makes pandas read each column in one pass so mixed types are inferred consistently.
df = pd.read_csv(csv_path, low_memory=False)
required_columns = ["SEGMENT_LENGTH_KM", "Shape__Length"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

print(f"Loaded: {csv_path}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print(df[required_columns].head())

Loaded: c:\Users\ti9593891\road-geometry-regression\data\road_segments.csv
Rows: 25,415 | Columns: 33
   SEGMENT_LENGTH_KM  Shape__Length
0              0.546     889.564188
1              0.050      82.665659
2              1.609    2619.785539
3              0.030      50.717404
4              0.638    1016.861700


## 3. Check the two measurements

Keep `df` as the complete loaded dataset. For modeling, convert the two measurement columns to numeric values and identify rows where either value is missing, nonnumeric, or infinite. The notebook reports these counts rather than silently dropping records.

In [28]:
# Build a numeric copy of the two columns and count problems, without altering df.
numeric = pd.DataFrame(index=df.index)
quality_rows = []

for column in required_columns:
    # errors="coerce" turns text that is not a number into NaN so it can be counted.
    converted = pd.to_numeric(df[column], errors="coerce")
    finite = pd.Series(np.isfinite(converted), index=df.index)
    numeric[column] = converted
    quality_rows.append(
        {
            "column": column,
            "missing in source": int(df[column].isna().sum()),
            "nonnumeric": int((df[column].notna() & converted.isna()).sum()),
            "nonfinite numeric": int((converted.notna() & ~finite).sum()),
            "finite numeric": int(finite.sum()),
        }
    )

print(pd.DataFrame(quality_rows).set_index("column"))

# A row is usable only if BOTH x and y are finite numbers (not NaN or infinity).
valid_pairs = np.isfinite(numeric[required_columns]).all(axis=1)
model_data = numeric.loc[valid_pairs].copy()
excluded_count = int((~valid_pairs).sum())
print(f"Eligible paired records: {len(model_data):,} of {len(df):,}")
print(f"Rows excluded from modeling due to invalid X or Y: {excluded_count:,}")

if model_data.empty:
    raise ValueError("No rows have finite values for both regression measurements.")

                   missing in source  nonnumeric  nonfinite numeric  \
column                                                                
SEGMENT_LENGTH_KM                  0           0                  0   
Shape__Length                      0           0                  0   

                   finite numeric  
column                             
SEGMENT_LENGTH_KM           25415  
Shape__Length               25415  
Eligible paired records: 25,415 of 25,415
Rows excluded from modeling due to invalid X or Y: 0


In [29]:
print(model_data.describe().T)

                     count         mean          std       min         25%  \
SEGMENT_LENGTH_KM  25415.0     1.060867     2.112148  0.003000    0.120000   
Shape__Length      25415.0  1735.508222  3761.816548  4.682682  194.626853   

                          50%          75%          max  
SEGMENT_LENGTH_KM    0.550000     1.627000     100.8060  
Shape__Length      885.732282  2564.073637  187320.3682  


## 4. Plot the relationship

Each point is one eligible road-segment record. Hover over a point to see its road name and kilometre range, drag the slider or scroll to zoom, and use the buttons to switch between linear and log axes (log axes spread out the many short segments that a linear axis squashes together).

In [30]:
x = model_data["SEGMENT_LENGTH_KM"]  # predictor (X)
y = model_data["Shape__Length"]  # target (Y)

# Hover details come from the full dataframe; model_data shares its index.
hover = df.loc[model_data.index, ["ROADNAME", "KM_FROM", "KM_TO"]]

scatter_option = charts.scatter_option(
    x,
    y,
    hover.to_numpy().tolist(),
    x_label=X_LABEL,
    y_label=Y_LABEL,
    title="Road segment length vs. GIS geometry length",
    subtitle=f"{len(x):,} segments. {UNIT_NOTE}",
)
charts.show(scatter_option, axis_toggle=True)

## 5. Fit an initial regression line

The model estimates `Ŷ = intercept + slope × X` and keeps the intercept, as recommended for the first model. This fit uses all eligible records to describe the overall relationship. Since those same records are used to fit the line, do not treat the fit or its coefficients as an unbiased evaluation of predictive performance.

In [31]:
# The C++ functions take plain NumPy arrays, so convert from pandas.
x_values = x.to_numpy()
y_values = y.to_numpy()

# Closed-form ordinary least squares, computed in C++.
intercept, slope = regression_cpp.fit_ols(x_values, y_values)
print(f"Intercept (b0): {intercept:.6g} GIS length units")
print(f"Slope (b1):     {slope:.6g} GIS length units per kilometre")
print(f"Fitted equation: predicted Shape__Length = {intercept:.6g} + {slope:.6g} * SEGMENT_LENGTH_KM")

Intercept (b0): -149.681 GIS length units
Slope (b1):     1777.03 GIS length units per kilometre
Fitted equation: predicted Shape__Length = -149.681 + 1777.03 * SEGMENT_LENGTH_KM


In [32]:
# Two points are enough to draw a straight line; 300 evenly spaced x values span the data range.
x_line = np.linspace(x.min(), x.max(), 300)
y_line = regression_cpp.predict(x_line, intercept, slope)
# R-squared: share of the variation in y explained by the line (1 = perfect). It is computed on
# the same data used to fit, so it describes the fit, not performance on new data.
r_squared = regression_cpp.r2(y_values, regression_cpp.predict(x_values, intercept, slope))

fit_option = charts.fit_option(
    x_values,
    y_values,
    x_line,
    y_line,
    x_label=X_LABEL,
    y_label=Y_LABEL,
    title="Initial linear fit (all eligible records)",
    subtitle=f"R² = {r_squared:.4f} on the fitting data, not a test score. {UNIT_NOTE}",
    equation=f"Fit: y = {intercept:,.1f} + {slope:,.1f}x",
)
charts.show(fit_option)

## 6. Hold out a test set

The fit above used every record, so its R² only describes the data it was fitted on. To estimate how the line behaves on segments it has not seen, shuffle the eligible records with a fixed seed, keep 80% for training, and set 20% aside as a test set that is used only for scoring.

In [33]:
# A fixed seed makes the shuffle repeatable: the same rows land in train and test every run.
SEED = 42
TEST_FRACTION = 0.2  # share of rows kept aside for testing

rng = np.random.default_rng(SEED)
order = rng.permutation(len(x_values))  # a random ordering of the row positions
n_test = int(round(len(order) * TEST_FRACTION))
# The first n_test shuffled positions become the test set; the rest are for training.
test_idx, train_idx = order[:n_test], order[n_test:]

x_train, y_train = x_values[train_idx], y_values[train_idx]
x_test, y_test = x_values[test_idx], y_values[test_idx]
print(f"Training rows: {len(x_train):,} | Test rows: {len(x_test):,} | seed={SEED}")

Training rows: 20,332 | Test rows: 5,083 | seed=42


## 7. Fit on training data, score on test data

The line is fitted on the training rows only. MAE and RMSE are in the same (unconfirmed) units as `Shape__Length`; R² is unitless. A test score close to the training score suggests the line is not overfitted.

In [34]:
# Learn the line from the training rows only; the test rows must stay unseen.
train_b0, train_b1 = regression_cpp.fit_ols(x_train, y_train)
print(f"Training fit: Shape__Length = {train_b0:.6g} + {train_b1:.6g} * SEGMENT_LENGTH_KM\n")

# MAE: average miss. RMSE: like MAE but punishes big misses more. R2: share of variation explained.
scores = {}
for name, xs, ys in [("train", x_train, y_train), ("test", x_test, y_test)]:
    pred = regression_cpp.predict(xs, train_b0, train_b1)
    scores[name] = {
        "MAE": regression_cpp.mae(ys, pred),
        "RMSE": regression_cpp.rmse(ys, pred),
        "R2": regression_cpp.r2(ys, pred),
    }

print(pd.DataFrame(scores).T.round(4))

Training fit: Shape__Length = -152.668 + 1779.79 * SEGMENT_LENGTH_KM

            MAE      RMSE      R2
train  145.6062  256.9327  0.9955
test   146.7631  233.5230  0.9953


## 8. Look at the residuals

A residual is actual minus predicted. Points far from the zero line are segments the straight line fits poorly. Residuals are shown for the test rows only.

In [35]:
# Residual = actual - predicted. Positive means the line predicted too low.
test_pred = regression_cpp.predict(x_test, train_b0, train_b1)
residuals = y_test - test_pred

# argsort sorts ascending; [::-1] reverses it so the biggest misses (either sign) come first.
worst = np.argsort(np.abs(residuals))[::-1][:5]
print("Five largest test residuals (length km, actual, predicted, residual):")
for i in worst:
    print(f"  {x_test[i]:8.3f}  {y_test[i]:12.1f}  {test_pred[i]:12.1f}  {residuals[i]:12.1f}")

charts.show(
    charts.residual_option(
        x_test,
        residuals,
        x_label=X_LABEL,
        y_label="Residual (actual - predicted, source units)",
        title="Residuals on the test set",
        subtitle=f"{len(x_test):,} test segments. {UNIT_NOTE}",
    )
)

Five largest test residuals (length km, actual, predicted, residual):
    51.453      100403.9       91422.7        8981.2
    67.700      122784.9      120338.9        2446.0
    16.760       31812.9       29676.6        2136.4
     7.889       12003.4       13888.1       -1884.6
    12.590       24044.4       22254.8        1789.5


### Compare actual and predicted values

Each point is one held-out test segment. The dashed diagonal is perfect prediction; points closer to it have smaller errors. Hover to see the actual value, prediction, and their difference (actual minus predicted).

In [36]:
charts.show(
    charts.actual_predicted_option(
        y_test,
        test_pred,
        x_label="Actual GIS length (source units)",
        y_label="Predicted GIS length (source units)",
        title="Predicted vs. actual GIS length (test set)",
        subtitle=f"{len(y_test):,} held-out segments. {UNIT_NOTE}",
    )
)

## 9. Compare with a through-origin fit

A segment of zero length should have zero GIS length, so a line forced through the origin (`Ŷ = slope × X`, no intercept) is a reasonable alternative. Both models are fitted on the training rows and scored on the same test rows.

In [37]:
origin_slope = regression_cpp.fit_through_origin(x_train, y_train)
print(f"Through-origin fit: Shape__Length = {origin_slope:.6g} * SEGMENT_LENGTH_KM\n")

# Through-origin model has no intercept, so predict() gets intercept = 0.
models = {
    "OLS with intercept": (train_b0, train_b1),
    "Through origin": (0.0, origin_slope),
}
comparison = {}
for name, (b0, b1) in models.items():
    pred = regression_cpp.predict(x_test, b0, b1)
    comparison[name] = {
        "MAE": regression_cpp.mae(y_test, pred),
        "RMSE": regression_cpp.rmse(y_test, pred),
        "R2": regression_cpp.r2(y_test, pred),
    }
print("Test-set scores:")
print(pd.DataFrame(comparison).T.round(4))

Through-origin fit: Shape__Length = 1751.67 * SEGMENT_LENGTH_KM

Test-set scores:
                         MAE      RMSE      R2
OLS with intercept  146.7631  233.5230  0.9953
Through origin      134.9556  262.9834  0.9940
